[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-09-tiny-diffusion-ddim-guidance.ipynb)

# Advanced Discussion 9 — noise schedules, a tiny diffusion model, DDIM and guidance

Reproduces the chapter 9 advanced discussion: linear versus cosine schedules, a complete toy DDPM over three seeds, DDIM with fewer steps, and classifier-free guidance.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: 2-D toy data. Training takes about 25 s per model on CPU; the full notebook a few minutes.

In [ ]:
%pip install -q scipy

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, math
from scipy import linalg
np.random.seed(0); torch.manual_seed(0)

## 1. noise schedules: linear vs cosine

In [ ]:
T = 1000
beta_lin = torch.linspace(1e-4, 0.02, T); abar_lin = torch.cumprod(1 - beta_lin, 0)
s = 0.008; steps = torch.arange(T + 1) / T
f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2; abar_cos = (f / f[0])[1:]
print("t     abar_linear  abar_cosine   SNR_linear   SNR_cosine   (SNR = abar/(1-abar))")
for t in (0, 100, 250, 500, 750, 999):
    al, ac = abar_lin[t].item(), abar_cos[t].item()
    print("%4d  %10.4f  %11.4f  %10.3g  %11.3g" % (t, al, ac, al / (1 - al), ac / (1 - ac + 1e-12)))

## 2. one line to noise an image: q(x_t | x_0)

In [ ]:
x0 = torch.randn(4, 2); eps = torch.randn_like(x0); t = 500
xt = abar_lin[t].sqrt() * x0 + (1 - abar_lin[t]).sqrt() * eps
print("signal weight %.3f, noise weight %.3f at t=%d" % (abar_lin[t].sqrt().item(), (1 - abar_lin[t]).sqrt().item(), t))

## 3. a complete toy DDPM on 8 Gaussians (data from Chapter 8)

In [ ]:
def sample_real(n, rng, labels=False):
    k = rng.randint(0, 8, n); ang = k * (2 * np.pi / 8)
    x = (2.0 * np.c_[np.cos(ang), np.sin(ang)] + 0.05 * rng.randn(n, 2)).astype(np.float32)
    return (x, k) if labels else x
CENTERS = 2.0 * np.c_[np.cos(np.arange(8) * 2 * np.pi / 8), np.sin(np.arange(8) * 2 * np.pi / 8)]
TS = 100
beta = torch.linspace(1e-4, 0.05, TS); alpha = 1 - beta; abar = torch.cumprod(alpha, 0)
class EpsNet(nn.Module):
    def __init__(s, n_cls=0, h=256):
        super().__init__(); s.n_cls = n_cls
        s.t_emb = nn.Sequential(nn.Linear(16, h), nn.SiLU())
        s.c_emb = nn.Embedding(n_cls + 1, h) if n_cls else None            # last index = "no label"
        s.net = nn.Sequential(nn.Linear(2 + h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(s, x, t, c=None):
        freqs = torch.exp(torch.linspace(0, math.log(100), 8))
        tt = t.float().unsqueeze(1) / TS * freqs.unsqueeze(0) * 2 * math.pi
        e = s.t_emb(torch.cat([tt.sin(), tt.cos()], 1))
        if s.c_emb is not None: e = e + s.c_emb(c)
        return s.net(torch.cat([x, e], 1))
def train_ddpm(cond, iters=6000, bs=256, seed=0, p_uncond=0.15):
    torch.manual_seed(seed); r = np.random.RandomState(seed)
    m = EpsNet(8 if cond else 0); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for it in range(iters):
        x0, k = sample_real(bs, r, labels=True); x0 = torch.tensor(x0)
        t = torch.randint(0, TS, (bs,)); eps = torch.randn_like(x0)
        xt = abar[t].sqrt().unsqueeze(1) * x0 + (1 - abar[t]).sqrt().unsqueeze(1) * eps
        c = None
        if cond:
            c = torch.tensor(k); c = torch.where(torch.rand(bs) < p_uncond, torch.full_like(c, 8), c)      # label dropout
        loss = F.mse_loss(m(xt, t, c), eps); opt.zero_grad(); loss.backward(); opt.step()
        if it in (0, 999, 5999): print("  iter %d loss %.3f" % (it, loss.item()))
    return m.eval()
@torch.no_grad()
def sample_ddpm(m, n, c=None, w=0.0):
    x = torch.randn(n, 2)
    for t in reversed(range(TS)):
        tt = torch.full((n,), t)
        if c is None: e = m(x, tt)
        else: e = (1 + w) * m(x, tt, c) - w * m(x, tt, torch.full_like(c, 8))        # classifier-free guidance
        mean = (x - beta[t] / (1 - abar[t]).sqrt() * e) / alpha[t].sqrt()
        x = mean + (beta[t].sqrt() * torch.randn_like(x) if t > 0 else 0)
    return x
@torch.no_grad()
def sample_ddim(m, n, n_steps):
    x = torch.randn(n, 2); ts = torch.linspace(TS - 1, 0, n_steps).long()
    for i, t in enumerate(ts):
        e = m(x, torch.full((n,), int(t))); x0 = (x - (1 - abar[t]).sqrt() * e) / abar[t].sqrt()
        ab_prev = abar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * e                                # deterministic DDIM step
    return x
def modes_quality(fake, thr=0.5):
    d = np.linalg.norm(fake[:, None] - CENTERS[None], axis=2); near = d.min(1) < thr
    counts = np.bincount(d.argmin(1)[near], minlength=8); return int((counts >= 0.01 * len(fake)).sum()), float(near.mean())
def frechet(a, b):
    m1, m2 = a.mean(0), b.mean(0); s_ = linalg.sqrtm(np.cov(a.T) @ np.cov(b.T)).real
    return float(((m1 - m2) ** 2).sum() + np.trace(np.cov(a.T) + np.cov(b.T) - 2 * s_))
real = sample_real(2000, np.random.RandomState(123))
t0 = time.time(); model = train_ddpm(cond=False); print("trained in %.0fs" % (time.time() - t0))
torch.manual_seed(1)
print("sampler        network calls  modes  quality  Frechet")
for name, fn, calls in (("DDPM (100)", lambda: sample_ddpm(model, 2000), 100), ("DDIM 50 steps", lambda: sample_ddim(model, 2000, 50), 50),
                        ("DDIM 20 steps", lambda: sample_ddim(model, 2000, 20), 20), ("DDIM 10 steps", lambda: sample_ddim(model, 2000, 10), 10), ("DDIM 5 steps", lambda: sample_ddim(model, 2000, 5), 5)):
    x = fn().numpy(); mo, q = modes_quality(x); print("%-14s %13d  %5d  %7.2f  %7.3f" % (name, calls, mo, q, frechet(real, x)))

## 3b. the same experiment as Chapter 8's GANs: three training seeds

In [ ]:
print("diffusion, three training seeds (DDPM sampler, 2000 samples each):")
for sd in (0, 1, 2):
    mm = train_ddpm(cond=False, seed=sd); torch.manual_seed(10 + sd); x = sample_ddpm(mm, 2000).numpy()
    mo, q = modes_quality(x); print("  seed %d: modes %d/8, quality %.2f, Frechet %.3f" % (sd, mo, q, frechet(real, x)))

## 4. classifier-free guidance on the labelled version

In [ ]:
t0 = time.time(); cmodel = train_ddpm(cond=True); print("conditional model trained in %.0fs" % (time.time() - t0))
torch.manual_seed(2)
lab = torch.tensor(np.repeat(np.arange(8), 250))
print("guidance w   label accuracy   spread inside a mode (std)")
for w in (0.0, 1.0, 3.0, 6.0):
    x = sample_ddpm(cmodel, len(lab), lab, w).numpy()
    nearest = np.linalg.norm(x[:, None] - CENTERS[None], axis=2).argmin(1)
    spread = np.mean([x[lab.numpy() == k].std(0).mean() for k in range(8)])
    print("%9.1f   %14.3f   %8.3f" % (w, (nearest == lab.numpy()).mean(), spread))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.